# Rummi Poker Grid ML 레벨링 워크벤치

이 노트북은 `tools/sim/ml_leveling_report.py`와 같은 분석 함수를 사용합니다.
현재 기준은 실제 ML 전환 완료가 아니라 규칙 기반 진단과 데이터 충분성 점검입니다.
tabular ML 셀은 향후 전환을 위한 실험 워크벤치이며, 런타임 밸런스를 자동 조정하지 않습니다.

- 첫 실행 셀은 필요한 라이브러리를 현재 커널에 설치합니다.
- 옵션은 다음 코드 셀 하나만 수정하면 됩니다.
- UI 위젯이 보이지 않는 환경에서는 같은 셀의 `OPTIONS` 딕셔너리 값을 바꾸면 됩니다.


In [ ]:
# 라이브러리 자동 확인/설치 셀입니다. 이 셀은 수정하지 않아도 됩니다.
from __future__ import annotations

import importlib
import os
import subprocess
import sys
from pathlib import Path

# 노트북 위치는 notebooks/ 이므로 repo root는 한 단계 위입니다.
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
elif (REPO_ROOT / "notebooks").exists():
    REPO_ROOT = REPO_ROOT

# 상대 경로 옵션이 항상 repo root 기준으로 동작하게 작업 폴더를 고정합니다.
os.chdir(REPO_ROOT)

# 현재 repo의 Python 모듈을 import하기 위해 루트 경로를 추가합니다.
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# import 이름과 pip package 이름이 다른 경우가 있어 map으로 관리합니다.
REQUIRED_IMPORTS = {
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "ipywidgets": "ipywidgets",
}

missing_packages = []
for import_name, package_name in REQUIRED_IMPORTS.items():
    try:
        importlib.import_module(import_name)
    except Exception:
        missing_packages.append(package_name)

if missing_packages:
    print("설치할 라이브러리:", missing_packages)
    # sys.executable은 현재 노트북 커널의 Python입니다.
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("필요한 라이브러리가 이미 준비되어 있습니다.")


In [ ]:
# EDIT_THIS_CELL = True
# 이 셀만 수정하면 됩니다. UI가 보이지 않으면 OPTIONS 값을 직접 바꾸세요.

OPTIONS = {
    # summary_json: Dart 시뮬레이터가 만든 summary JSON 경로입니다.
    "summary_json": "logs/sim/planner_v2_ml_label_v1_preview_100_summary.json",
    # out_dir: ML 워크벤치 리포트와 차트를 저장할 폴더입니다.
    "out_dir": "logs/sim",
    # leveling_goal: 어떤 방향으로 밸런스를 볼지 정합니다.
    # - overall: 전체 상태를 넓게 봅니다.
    # - onboarding: 초반 유입과 쉬운/어려운 구간을 봅니다.
    # - boss_wall: 보스 벽과 too_hard 구간을 봅니다.
    # - tempo: 느린 클리어와 tempo_drag를 봅니다.
    "leveling_goal": "overall",
    # target_labels: 함께 참고할 규칙 기반 label입니다. 모델 target은 leveling_loss입니다.
    "target_labels": ["too_hard", "tempo_drag", "good_playfeel"],
    # top_n: 각 섹션에서 보여줄 후보 수입니다.
    "top_n": 12,
    # auto_install_missing: 노트북 실행 중 누락 라이브러리를 자동 설치할지 여부입니다.
    "auto_install_missing": True,
}

# 아래 UI는 선택 사항입니다. UI가 렌더링되면 값 변경 후 다음 셀을 실행하세요.
try:
    import ipywidgets as widgets
    from IPython.display import display

    goal_widget = widgets.Dropdown(
        options=["overall", "onboarding", "boss_wall", "tempo"],
        value=OPTIONS["leveling_goal"],
        description="목표",
    )
    top_n_widget = widgets.IntSlider(
        value=OPTIONS["top_n"], min=5, max=30, step=1, description="Top N"
    )
    summary_widget = widgets.Text(
        value=OPTIONS["summary_json"], description="Summary", layout=widgets.Layout(width="90%")
    )
    display(summary_widget, goal_widget, top_n_widget)
except Exception as error:
    print("UI 위젯을 표시하지 못했습니다. OPTIONS 딕셔너리를 직접 수정하세요:", error)


In [ ]:
# UI 위젯 값이 있으면 OPTIONS에 반영합니다. 없으면 OPTIONS 딕셔너리를 그대로 씁니다.
try:
    OPTIONS["summary_json"] = summary_widget.value
    OPTIONS["leveling_goal"] = goal_widget.value
    OPTIONS["top_n"] = int(top_n_widget.value)
except Exception:
    pass

print("사용 옵션:")
for key, value in OPTIONS.items():
    print(f"- {key}: {value}")


In [ ]:
# 공통 Python 리포터를 호출합니다. CLI와 노트북이 같은 함수를 사용합니다.
import os
from pathlib import Path

from IPython.display import Markdown, display

from tools.sim.ml_leveling_report import run_from_options

result = run_from_options(OPTIONS)
print("생성된 리포트:", result["report_path"])
print("생성된 차트:")
for chart_path in result["chart_paths"]:
    print("-", chart_path)
for warning in result["warnings"]:
    print("경고:", warning)

# 저장된 리포트 파일은 logs/sim 안의 PNG를 같은 폴더 기준으로 참조합니다.
# 노트북 화면에서는 notebook 파일 위치 기준 URL이 필요해서 차트 경로만 변환합니다.
markdown_text = Path(result["report_path"]).read_text(encoding="utf-8")
notebook_dir = REPO_ROOT / "notebooks"
for chart_path in result["chart_paths"]:
    chart_path_obj = Path(chart_path)
    notebook_relative_path = Path(
        os.path.relpath(chart_path_obj, start=notebook_dir)
    ).as_posix()
    markdown_text = markdown_text.replace(
        f"]({chart_path_obj.name})",
        f"]({notebook_relative_path})",
    )

display(Markdown(markdown_text))
